In [374]:
import pandas as pd
from IPython.display import display, clear_output
import ipywidgets as widgets
import json

### Set up dropdowns

In [375]:
with open('../synthetic_data/motherboards.json', 'r') as file:
    mobo_data = json.load(file)

mobo_choices = [(f"{mobo['brand']} {mobo['model']}", mobo['id']) for mobo in mobo_data]

In [376]:
# everyone but mobo has id + name
# function time baby

def get_component_choices(component):
    with open(f'../synthetic_data/{component}.json', 'r') as file:
        component_data = json.load(file)
    return [(f"{item['name']}", item['id']) for item in component_data]

In [377]:
# also all components need the same stuff for the dropdowns, so let's make a function for that too

def create_dropdown(component, choices=None, display_it=False):
    choices = choices if choices is not None else get_component_choices(component)
    dropdown = widgets.Select(
        options=choices,
        value=choices[0][1],
        description=f'{component}:',
        disabled=False,
        layout=widgets.Layout(width='max-content'),
        rows=len(choices),
    )
    if display_it == True: display(dropdown)
    return dropdown

# note to future self: mobo_dropdown.value to get selected ID, mobo_dropdown.label to get selected label

In [378]:
mobo_dropdown = create_dropdown('motherboards', mobo_choices)

component_list = ["cases", "coolers", "cpus", "fans", "gpus", "lighting", "psus", "ram", "storage"]
dropdowns = [create_dropdown(component) for component in component_list]
dropdowns.insert(0, mobo_dropdown)

I want to check whether components are compatible. There are a couple of ways to do this. I could poll with every selection, or I can just provide a big fat button so you submit the build, then check. 

Performance wise, the latter wins, but I don't think any of this is performance intensive enough to warant a hack. 

Ok, gonna go for polling. 

I also think that gives more opportunity for learning.

In [379]:
# add an observer for every dropdown
def on_dropdown_change(change):
    values = [dropdown.value for dropdown in dropdowns]
    build_text.value = f"{newline.join([dropdown.value for dropdown in dropdowns])}"

for dropdown in dropdowns:
    dropdown.observe(on_dropdown_change, names="value")

In [380]:
newline = "\n"
build_text = widgets.Textarea(
    value=f"{newline.join([dropdown.value for dropdown in dropdowns])}",
    placeholder='Type something',
    description='Build:',
    disabled=True,
    rows=len(dropdowns),
    layout=widgets.Layout(justify_content='flex-end', width='max-content')
)

In [381]:
widgets.GridBox(
    children=dropdowns + [build_text],
    layout=widgets.Layout(
        # width='50%',
        grid_template_columns='auto auto auto',
        grid_template_rows='auto auto auto auto', 
        grid_gap='5px 10px',
    )
)

GridBox(children=(Select(description='motherboards:', layout=Layout(width='max-content'), options=(('Fictional…